# 02 · Silver · `medidores`
### V-Energy Grid 360 — Arquitetura Medalhão

Lê `v_energy.bronze.medidores` (`medidores.csv`, tudo como texto) e grava `v_energy.silver.medidores` com os dados limpos, tipados e confiáveis para a Gold. **Grão: 1 linha = 1 medidor** (uma UC pode ter um medidor RETIRADO e um ATIVO, depois de uma troca).

**Fonte das regras:** `exploracao_medidores.ipynb` e a seção `medidores` do `docs/data_quality_log.md`.

| # | Tratamento | Linhas afetadas | Regra | Log |
|---|---|---|---|---|
| 1 | Cópias exatas de linhas | 513 | `dropDuplicates(["id_medidor"])` | M3 |
| 2 | Sentinela `9999-12-31` em `data_retirada` | 13.500 | vira nulo; demais datas viram `date` | M2 |
| 3 | `data_instalacao` em 4 formatos | 3.694 fora do ISO | cada padrão com o seu formato (hífen = **mês primeiro**) → `date` | M4 |
| 4 | `data_ultima_manutencao` 100% vazia | 51.299 | mantida como `date`, sem uso em métricas | M1 |
| 5 | `tipo` com 10 grafias | 5.297 | normalizar para `ELETRONICO`, `ELETROMECANICO`, `SMART` | M5 |
| 6 | `modelo` com espaços | 3.078 | `trim` | M6 |

**Ordem das operações (importa):**
1. Remover as cópias antes de qualquer contagem.
2. Converter a sentinela **antes** do cast de data e ler o padrão de `data_instalacao` do **texto original**.
3. Nenhum medidor é removido além das cópias: todos os 51.299 são usados em `silver.leituras` (exploração, seção 10).

**Resultado esperado:** 51.299 linhas, com `id_medidor` único e no máximo 1 medidor ATIVO por UC.

## 1. Parâmetros do ambiente

In [0]:
catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"
landing_path = f"/Volumes/{catalog}/bronze/landing"

from pyspark.sql import functions as F, Row
from datetime import datetime

tabela_bronze = f"{bronze_schema}.medidores"
tabela_silver = f"{silver_schema}.medidores"

def n_br(n):
    return f"{n:,}".replace(",", ".")

print(f"origem:  {tabela_bronze}")
print(f"destino: {tabela_silver}")

## 2. Funções de checagem de qualidade

- `dq_check(table_name, check_name, df, condition)`: `condition` é a condição **esperada**. Falham as linhas que a violam.
- `dq_check_unique(table_name, check_name, df, key_cols)`: passa se a chave não se repete.

As funções só acumulam os resultados em `dq_results`. Depois das checagens, o notebook **grava o resultado em `silver.dq_log`** (sempre) e **só então decide se grava a Silver** (só se todas passarem).

**Cuidado com nulos:** linhas em que a condição resulta em nulo não contam como falha. Quando o nulo também for um problema, há um `dq_check` separado com `isNotNull()`.

In [0]:
dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

def dq_check_contagem(table_name: str, check_name: str, df, esperado: int):
    """Checagem de contagem total: passa só se o número de linhas for exatamente o esperado."""
    total = df.count()
    diferenca = abs(total - esperado)
    passed = diferenca == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=diferenca, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {n_br(total)} linhas (esperado {n_br(esperado)})")

## 3. Leitura da Bronze

Leitura sem transformação. Conferências: **51.812** linhas e todas as colunas como `string`. O DataFrame `bronze` fica guardado para a comparação Bronze × Silver no fim do notebook.

In [0]:
bronze = spark.table(tabela_bronze)

n_bronze = bronze.count()
print(f"Linhas na Bronze: {n_br(n_bronze)}")  # esperado: 51.812

bronze.printSchema()
display(bronze.limit(5))

## 4. Cópias exatas de linhas (log M3)

**Decisão:** `dropDuplicates(["id_medidor"])`.

**Evidência:** 51.812 linhas para 51.299 `id_medidor` distintos. Sem as colunas de auditoria, as linhas distintas também são 51.299, e nenhum id tem conteúdo diferente entre as repetições. São 513 ids que aparecem exatamente 2 vezes, todos idênticos, então manter um deles não perde informação.

**Conferência:** 513 linhas removidas; restam 51.299.

In [0]:
# Tratamento 1 · cópias exatas (esperado: 513 removidas, 51.299 restantes)
df_dedup = bronze.dropDuplicates(["id_medidor"])
n_dedup = df_dedup.count()
print(f"Cópias removidas: {n_br(n_bronze - n_dedup)} | antes: {n_br(n_bronze)} | depois: {n_br(n_dedup)}")

## 5. `data_retirada`: sentinela `9999-12-31` (log M2)

**Decisão:** a sentinela vira nulo real; as datas reais viram `date`.

**Evidência:** a sentinela aparece em 13.500 medidores, **todos ATIVOS**. Os 6.299 RETIRADOS têm data real (todas em ISO), e nenhum ATIVO tem data real. Ou seja, `9999-12-31` significa "não retirado", exatamente como o nulo dos outros 31.500 ativos.

**Conferência:** 13.500 sentinelas; depois da conversão, 6.299 datas e 45.000 nulos, e nenhuma data real perdida.

In [0]:
# Tratamento 2 · sentinela 9999-12-31 vira nulo; datas reais viram date
n_sentinela = df_dedup.filter(F.col("data_retirada") == "9999-12-31").count()

df_retirada = df_dedup.withColumn(
    "data_retirada",
    F.when(F.col("data_retirada") == "9999-12-31", F.lit(None).cast("date"))
     .otherwise(F.expr("to_date(try_to_timestamp(data_retirada, 'yyyy-MM-dd'))")),
)

# esperado: 13.500 sentinelas | 6.299 datas | 45.000 nulos | 0 datas reais que não converteram
print("sentinelas convertidas em nulo:", n_br(n_sentinela))
print("data_retirada preenchida:      ", n_br(df_retirada.filter("data_retirada IS NOT NULL").count()))
print("data_retirada nula:            ", n_br(df_retirada.filter("data_retirada IS NULL").count()))
perdidas = (df_dedup.filter(F.col("data_retirada").isNotNull() & (F.col("data_retirada") != "9999-12-31"))
            .join(df_retirada.filter("data_retirada IS NULL").select("id_medidor"), "id_medidor", "left_semi"))
print("datas reais que não converteram:", n_br(perdidas.count()))

## 6. `data_instalacao` em quatro formatos (log M4)

**Decisão:** converter cada padrão com o seu formato e gravar como `date`.

| Padrão | Formato | Linhas |
|---|---|---|
| `9999-99-99` | `yyyy-MM-dd` | 47.605 |
| `99-99-9999` | `MM-dd-yyyy` (**mês primeiro**) | 1.250 |
| `99/99/9999` | `dd/MM/yyyy` | 1.235 |
| `99.99.9999` | `dd.MM.yyyy` | 1.209 |

**Evidência da ordem dia/mês (3 testes independentes, exploração seção 7):**
1. **Partes maiores que 12:** no hífen, 755 linhas têm a 2ª parte > 12 (só pode ser dia); na barra e no ponto, 723 e 726 têm a 1ª parte > 12.
2. **Data de ligação da UC:** o formato proposto bate com a ligação em ~87% dos não ISO, a mesma taxa dos medidores em ISO (87,6%). A ordem trocada bate em no máximo 45 linhas.
3. **Troca de medidor:** o mês de instalação do medidor novo é o mês da troca em 142 de 142, 142 de 142 e 173 de 173.

É o mesmo mapeamento de `leituras` (L6) e `faturas` (F4). **Risco evitado:** um formato único converteria datas para o mês errado, sem dar erro.

**Conferência:** nenhuma data nula depois da conversão.

In [0]:
# Tratamento 3 · data_instalacao em 4 formatos: cada padrão com o seu formato (hífen = MÊS primeiro)
formatos = {
    "9999-99-99": "yyyy-MM-dd",
    "99-99-9999": "MM-dd-yyyy",
    "99.99.9999": "dd.MM.yyyy",
    "99/99/9999": "dd/MM/yyyy",
}

data_convertida = None
for padrao, fmt in formatos.items():
    cond = F.col("padrao_data") == padrao
    valor = F.expr(f"to_date(try_to_timestamp(data_instalacao, '{fmt}'))")
    data_convertida = F.when(cond, valor) if data_convertida is None else data_convertida.when(cond, valor)

df_instalacao = (
    df_retirada
    .withColumn("padrao_data", F.regexp_replace("data_instalacao", "[0-9]", "9"))  # lê o texto original
    .withColumn("data_instalacao", data_convertida)
)

# esperado: ISO 47.605 | 99-99-9999 1.250 | 99/99/9999 1.235 | 99.99.9999 1.209
display(df_instalacao.groupBy("padrao_data").count().orderBy(F.desc("count")))
# esperado: 0
print("data_instalacao nula após a conversão:", n_br(df_instalacao.filter("data_instalacao IS NULL").count()))

## 7. `data_ultima_manutencao` 100% vazia (log M1)

**Decisão:** manter a coluna, convertida para `date` (fica toda nula), e **não usar em métricas**.

**Por que manter:** o contrato da tabela fica estável. Se o sistema de origem passar a enviar a data, a Silver já a trata sem mudança de schema.

**Impacto na Gold:** para avaliar o medidor (por exemplo, na fila de perdas), usar a **idade** calculada a partir de `data_instalacao`, e não a manutenção.

**Conferência:** 51.299 nulos.

In [0]:
# Tratamento 4 · data_ultima_manutencao: mantida como date, sem uso em métricas
df_manutencao = df_instalacao.withColumn(
    "data_ultima_manutencao", F.expr("to_date(try_to_timestamp(data_ultima_manutencao, 'yyyy-MM-dd'))")
)
# esperado: 51.299 (100%)
print("data_ultima_manutencao nula:", n_br(df_manutencao.filter("data_ultima_manutencao IS NULL").count()))

## 8. `tipo` com dez grafias (log M5)

**Decisão:** normalizar para três valores. Primeiro `upper(trim)`, depois tirar acentos e hífen, e por fim mapear:

| Tipo na Silver | Grafias na Bronze |
|---|---|
| `ELETRONICO` | `ELETRONICO`, `Eletronico`, `eletrônico` |
| `ELETROMECANICO` | `ELETROMECANICO`, `ELETRO-MECANICO`, `Eletromecanico`, `eletromecânico` |
| `SMART` | `SMART`, `Smart`, `smart meter` |

Qualquer grafia fora do mapa vira nulo **de propósito**: a checagem "tipo não nulo" falha e avisa que apareceu uma grafia nova.

**Evidência:** depois da normalização, cada modelo tem um único tipo (checagem na seção 11).

**Conferência:** 5.297 linhas mudaram; totais de 26.339 eletrônicos, 16.493 eletromecânicos e 8.467 smart.

In [0]:
# Tratamento 5 · tipo: upper(trim) -> sem acento -> sem hífen -> mapa para 3 valores
tipo_limpo = F.regexp_replace(
    F.translate(F.upper(F.trim(F.col("tipo"))), "ÁÂÃÉÊÍÓÔÕÚÇ", "AAAEEIOOOUC"), "-", ""
)
mapa_tipo = {
    "ELETRONICO": "ELETRONICO",
    "ELETROMECANICO": "ELETROMECANICO",
    "SMART": "SMART",
    "SMART METER": "SMART",
}
tipo_final = None
for origem, destino in mapa_tipo.items():
    tipo_final = (F.when(tipo_limpo == origem, F.lit(destino)) if tipo_final is None
                  else tipo_final.when(tipo_limpo == origem, F.lit(destino)))
# grafia fora do mapa vira nulo (sem otherwise) e é pega pela checagem de não nulo

df_tipo = df_manutencao.withColumn("tipo_original", F.col("tipo")).withColumn("tipo", tipo_final)

# esperado: 5.297 alteradas | ELETRONICO 26.339 | ELETROMECANICO 16.493 | SMART 8.467 | nulo 0
print("linhas com tipo alterado:", n_br(df_tipo.filter(F.col("tipo") != F.col("tipo_original")).count()))
display(df_tipo.groupBy("tipo").count().orderBy(F.desc("count")))
display(df_tipo.groupBy("tipo_original", "tipo").count().orderBy("tipo", F.desc("count")))

## 9. `modelo` com espaços e padronização de `status` (log M6)

**Decisão:** `trim` em `modelo`. São sempre os mesmos 9 modelos, com espaços no início e/ou no fim em 3.078 linhas, e o espaço não tem significado.

`status` não tinha variações (só `ATIVO` e `RETIRADO`); `upper(trim)` só garante a padronização, como em `tipo_leitura` na Silver de leituras.

**Conferência:** 3.078 modelos alterados; 9 modelos distintos; `status` com 45.000 ATIVO e 6.299 RETIRADO.

In [0]:
# Tratamento 6 · modelo com trim; status padronizado
df_texto = (
    df_tipo
    .withColumn("modelo_original", F.col("modelo"))
    .withColumn("modelo", F.trim(F.col("modelo")))
    .withColumn("status", F.upper(F.trim(F.col("status"))))
)

# esperado: 3.078 alterados | 9 modelos distintos | ATIVO 45.000 | RETIRADO 6.299
print("modelos alterados pelo trim:", n_br(df_texto.filter(F.col("modelo") != F.col("modelo_original")).count()))
print("modelos distintos:", df_texto.select("modelo").distinct().count())
display(df_texto.groupBy("status").count())

## 10. Seleção final das colunas

Colunas auxiliares (`padrao_data`, `tipo_original`, `modelo_original`) ficam de fora. A coluna de auditoria `source_file` é mantida, como na Silver de leituras.

In [0]:
df_silver = df_texto.select(
    "id_medidor", "id_uc", "modelo", "tipo", "status",
    "data_instalacao", "data_retirada", "data_ultima_manutencao",
    "source_file",
)
df_silver.printSchema()

## 11. Checagens de qualidade

Rodam sobre o resultado, **antes** de gravar. Cada uma protege uma decisão:

| Checagem | Protege |
|---|---|
| Total de linhas = 51.299 | cópias removidas na quantidade esperada (vale para esta extração) |
| `id_medidor` único | cópias (M3) |
| Colunas-chave não nulas | conversões de data (M4) e mapa de tipo (M5) |
| `id_medidor` e `id_uc` no padrão | formato dos ids |
| `tipo`, `status` e `modelo` em domínio válido | normalizações (M5, M6) |
| Cada modelo com o tipo esperado | mapa de tipo (M5) |
| RETIRADO ⇔ `data_retirada` preenchida | sentinela (M2) |
| Retirada não anterior à instalação; datas até o fim da extração | conversões de data (M2, M4) |
| No máximo 1 medidor ATIVO por UC | coerência do cadastro (visão 360) |
| `id_uc` existe em `unidades_consumidoras` | integridade referencial |
| Todo medidor usado em `silver.leituras` existe aqui | a Silver de leituras não fica com órfãs (L2) |

A checagem de contagem total é uma proteção desta extração única: se a Silver for reexecutada com dados novos, o valor esperado precisa ser atualizado.

In [0]:
# Checagens de qualidade antes de gravar (condição = o que deve ser verdade em todas as linhas)
t = "silver.medidores"
dq_results.clear()  # se esta célula for reexecutada, não acumula resultados antigos

# Volume e chave
dq_check_contagem(t, "total de linhas = 51.299", df_silver, 51_299)
dq_check_unique(t, "id_medidor único", df_silver, ["id_medidor"])

# Nulos
for coluna in ["id_medidor", "id_uc", "modelo", "tipo", "status", "data_instalacao"]:
    dq_check(t, f"{coluna} não nulo", df_silver, F.col(coluna).isNotNull())

# Formatos e domínios
dq_check(t, "id_medidor no padrão MD + 6 dígitos", df_silver, F.col("id_medidor").rlike("^MD[0-9]{6}$"))
dq_check(t, "id_uc no padrão UC + 6 dígitos", df_silver, F.col("id_uc").rlike("^UC[0-9]{6}$"))
dq_check(t, "tipo só ELETRONICO, ELETROMECANICO ou SMART", df_silver,
         F.coalesce(F.col("tipo").isin("ELETRONICO", "ELETROMECANICO", "SMART"), F.lit(False)))
dq_check(t, "status só ATIVO ou RETIRADO", df_silver,
         F.coalesce(F.col("status").isin("ATIVO", "RETIRADO"), F.lit(False)))

tipo_por_modelo = {
    "CG-500": "ELETROMECANICO", "EM-1200": "ELETROMECANICO", "EM-1400": "ELETROMECANICO",
    "EL-2200": "ELETRONICO", "EL-3000": "ELETRONICO", "NX-700": "ELETRONICO",
    "IQ-900": "SMART", "SM-4100": "SMART", "SM-4500": "SMART",
}
dq_check(t, "modelo entre os 9 conhecidos", df_silver,
         F.coalesce(F.col("modelo").isin(list(tipo_por_modelo)), F.lit(False)))
tipo_esperado = None
for modelo, tipo in tipo_por_modelo.items():
    tipo_esperado = (F.when(F.col("modelo") == modelo, F.lit(tipo)) if tipo_esperado is None
                     else tipo_esperado.when(F.col("modelo") == modelo, F.lit(tipo)))
dq_check(t, "cada modelo com o tipo esperado", df_silver,
         F.coalesce(F.col("tipo") == tipo_esperado, F.lit(False)))

# Datas e status
dq_check(t, "RETIRADO se e somente se data_retirada preenchida", df_silver,
         (F.col("status") == "RETIRADO") == F.col("data_retirada").isNotNull())
dq_check(t, "data_retirada não anterior à data_instalacao", df_silver,
         F.col("data_retirada").isNull() | (F.col("data_retirada") >= F.col("data_instalacao")))
dq_check(t, "data_instalacao até 2026-06-30 (fim da extração)", df_silver,
         F.col("data_instalacao") <= F.lit("2026-06-30").cast("date"))

# Coerência do cadastro
dq_check_unique(t, "no máximo 1 medidor ATIVO por UC", df_silver.filter("status = 'ATIVO'"), ["id_uc"])

# Integridade referencial com unidades_consumidoras (Bronze sem o cabeçalho repetido, log U1)
ucs = (spark.table(f"{bronze_schema}.unidades_consumidoras")
       .filter(F.col("id_uc") != "id_uc").select("id_uc").distinct().withColumn("uc_existe", F.lit(True)))
dq_check(t, "id_uc existe em unidades_consumidoras", df_silver.join(ucs, "id_uc", "left"),
         F.coalesce(F.col("uc_existe"), F.lit(False)))

# Todo medidor usado em silver.leituras precisa existir aqui (protege a regra de órfãs L2)
if spark.catalog.tableExists(f"{silver_schema}.leituras"):
    ids_leit = spark.table(f"{silver_schema}.leituras").select("id_medidor").distinct()
    ids_med = df_silver.select("id_medidor").withColumn("existe", F.lit(True))
    dq_check(t, "todo id_medidor de silver.leituras existe em medidores",
             ids_leit.join(ids_med, "id_medidor", "left"), F.coalesce(F.col("existe"), F.lit(False)))
else:
    print("[AVISO] silver.leituras ainda não existe: checagem de leituras pulada")

## 12. Registro das checagens em `silver.dq_log`

O resultado é gravado **sempre**, mesmo quando alguma checagem falha. Modo `append`: cada execução acrescenta linhas, identificadas por `checked_at`. A célula mostra só a execução atual.

In [0]:
# Persistência do dq_log (modo append, igual à aula); sempre grava, inclusive quando há falha
df_dq_log = spark.createDataFrame(dq_results)
df_dq_log.write.format("delta").mode("append").option("mergeSchema", "true") \
    .saveAsTable(f"{silver_schema}.dq_log")

display(
    spark.table(f"{silver_schema}.dq_log")
    .filter(F.col("table_name") == "silver.medidores")
    .orderBy(F.col("checked_at").desc())
    .limit(len(dq_results))
)

## 13. Gravação da Silver, só se todas as checagens passarem

Se qualquer checagem falhar, a tabela **não é gravada**: o notebook lista as falhas e é interrompido com erro. Se `silver.medidores` já existia, ela permanece como estava.

In [0]:
# Gravação da Silver: só acontece se todas as checagens passaram
falhas = [r for r in dq_results if not r.passed]

if falhas:
    print(f"{len(falhas)} checagem(ns) falharam. {tabela_silver} NÃO foi gravada:")
    for r in falhas:
        print(f"  - {r.check_name}: {n_br(r.failed_rows)} falha(s) em {n_br(r.total_rows)} linhas")
    raise Exception(f"{tabela_silver} não gravada: {len(falhas)} checagem(ns) de qualidade falharam (veja silver.dq_log)")

(
    df_silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_silver)
)
print(f"Todas as {len(dq_results)} checagens passaram. {tabela_silver} gravada: "
      f"{n_br(spark.table(tabela_silver).count())} linhas")

## 14. Bronze × Silver, com os mesmos medidores

Para cada tratamento, os `id_medidor` são escolhidos e mostrados nas duas tabelas, para ver o **texto original** da Bronze (`9999-12-31`, `03-15-2019`, `eletrônico`, `"  EL-3000 "`) ao lado do valor tratado. No caso das cópias, a Bronze mostra duas linhas e a Silver, uma.

In [0]:
# Bronze × Silver: os MESMOS id_medidor nas duas tabelas, um grupo de exemplos por tratamento
silver_t = spark.table(tabela_silver)
colunas_bronze = ["id_medidor", "id_uc", "modelo", "tipo", "status",
                  "data_instalacao", "data_retirada", "data_ultima_manutencao"]

def comparar(titulo, ids_df, n=3):
    ids = [r["id_medidor"] for r in ids_df.select("id_medidor").limit(n).collect()]
    print(f"=== {titulo} | ids: {ids}")
    print("Bronze (texto original):")
    display(bronze.filter(F.col("id_medidor").isin(ids)).select(*colunas_bronze).orderBy("id_medidor"))
    print("Silver (tratado):")
    display(silver_t.filter(F.col("id_medidor").isin(ids)).orderBy("id_medidor"))

# 1. Cópias: 2 linhas idênticas na Bronze, 1 na Silver
comparar("Cópias exatas", bronze.groupBy("id_medidor").count().filter("count > 1"), n=2)

# 2. Sentinela 9999-12-31 vira nulo
comparar("Sentinela em data_retirada", bronze.filter(F.col("data_retirada") == "9999-12-31"), n=2)

# 3. data_instalacao: um exemplo de cada formato fora do ISO
ids_datas = (bronze.withColumn("padrao", F.regexp_replace("data_instalacao", "[0-9]", "9"))
             .filter("padrao != '9999-99-99'").dropDuplicates(["padrao"]))
comparar("data_instalacao fora do ISO (um de cada formato)", ids_datas, n=3)

# 4. tipo: um exemplo de cada grafia fora do padrão
ids_tipo = (bronze.filter(~F.col("tipo").isin("ELETRONICO", "ELETROMECANICO", "SMART"))
            .dropDuplicates(["tipo"]))
comparar("tipo fora do padrão (uma de cada grafia)", ids_tipo, n=7)

# 5. modelo com espaços
comparar("modelo com espaços", bronze.filter(F.col("modelo") != F.trim("modelo")), n=3)